# PM 6 Beginner Tutorial: Cleaning CityConnect Data

Welcome! This notebook will guide you through your first complete data-preparation project.

You do **not** need to know everything before starting. Read each explanation, run the example code, look at the result and then complete the small activity.

## What is the goal?

CityConnect receives municipal service requests. Management wants to know where delays happen, but the data is messy. Before analysing it, you must make it clean and trustworthy.

## Important rule

Do not manually edit the CSV or JSON files. Every correction must happen through Python code so that another person can repeat your work.


# Step 0 — Set up Python

A **library** is a collection of ready-made tools.

- `pandas` helps us work with tables.
- `NumPy` helps us work with numbers and missing values.
- `Path` helps us refer to files and folders.

Run the cell below by clicking inside it and pressing **Shift + Enter**.


In [4]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)

DATA_FOLDER = Path(".")
OUTPUT_FOLDER = DATA_FOLDER / "outputs"
OUTPUT_FOLDER.mkdir(exist_ok=True)

print("Setup complete.")


Setup complete.


## Checkpoint

You should see:

`Setup complete.`

If you see an error, make sure the notebook and all three raw data files are inside the same folder.


# Step 1 — Frame the problem (PA0101)

Before touching the data, explain the real-world problem.

A good problem statement answers:

1. What is going wrong?
2. Who is affected?
3. What decision must be made?
4. What information is needed?

### Example structure

> CityConnect management is concerned that some municipal service requests take too long to resolve. This may reduce resident satisfaction and make it difficult to allocate staff. The analysis must identify where delays occur and first determine whether the available data is reliable enough.

### Your activity

Write your own version below. Also identify:
- the client;
- two stakeholders;
- the decision management wants to make;
- five measurable questions.


## My problem statement


City connect collects residents' information for service delivery in the city. This helps operations at the municipal level. This needs accurate information to allocate resources and personnel accordingly. However, if the data at hand has issues like missing values, duplicates, inconsistencies, incorrect spellings, wrong numerical values this will affect decision making in the report and the reliability of information in reporting.

The problem is to clean, validate and integrate the three different datasets so that management is able to analyse service demand vs performance, look at different demographics and demands, this said accurate data is needed to ensure correct decisions are made and that there is less leakages, therefore accurate data is needed for all of this to

## Stakeholders

_Write your answer here._
1. service centre managers
2. municipal leadership
3. customer service teams
4. Residence
5. IT/ Data science /Analytics

## Five measurable questions

1.  What is the average time for service resolution and how many remain unresolved after average handling time
2.  Is there any relationship between time and resident satisfaction, areas to improve
3.  Does each service centre have enough staff capacity to service its own demand
4.  What are common traits in terms language, education level devices used
5.  Which places receives highest service request, what normally leads to complains


# Step 2 — Load the raw data (PA0102)

To analyse data, we first load it into a **DataFrame**.

A DataFrame is like an Excel table inside Python.

We have three source files:

1. `service_requests_raw.csv`
2. `resident_profiles_raw.csv`
3. `service_centres_raw.json`

CSV files can be loaded with `pd.read_csv()`.


In [5]:
requests_raw = pd.read_csv(DATA_FOLDER / "service_requests_raw.csv")
residents_raw = pd.read_csv(DATA_FOLDER / "resident_profiles_raw.csv")

print("Requests loaded:", requests_raw.shape)
print("Residents loaded:", residents_raw.shape)


Requests loaded: (800, 14)
Residents loaded: (400, 8)


JSON files may have information nested inside a larger structure. We first read the JSON and then select the `centres` section.


In [6]:
import json

with open(DATA_FOLDER / "service_centres_raw.json", "r", encoding="utf-8") as file:
    centres_json = json.load(file)

centres_raw = pd.json_normalize(centres_json["centres"])

print("Centres loaded:", centres_raw.shape)


Centres loaded: (25, 8)


## Checkpoint

You should now have three DataFrames:
- `requests_raw`
- `residents_raw`
- `centres_raw`

Use `head()` to see the first five rows.


In [7]:
display(requests_raw.head())
display(residents_raw.head())
display(centres_raw.head())


,request_id,resident_id,created_at,resolved_at,service_type,submission_channel,priority,status,resolution_hours,satisfaction_rating,latitude,longitude,device_id,source_system
0,SR00001,R0355,2026-05-26 15:17,2026-05-27 11:00,Water,Web Portal,High,Resolved,19.73,NaN,-33.913755,31.024099,Device-003,CRM
1,SR00002,R0301,2026-02-06 20:52,NaN,Roads,Walk-in,High,In Progress,NaN,5,-32.107892,29.734713,Device-070,Mobile API
2,SR00003,R0197,2026-02-05 20:44,2026-02-08 18:51,Water,Mobile App,Medium,Closed,70.12,1,-33.350134,21.480373,Device-044,Call Centre Export
3,SR00004,R0394,2026-06-07 17:25,NaN,Waste,Walk-in,Low,In Progress,NaN,4,-29.931750,23.174613,Device-056,CRM
4,SR00005,R0349,03/02/2026 09:15,NaN,Water,Call Centre,High,In Progress,NaN,2,-27.995956,19.423921,Device-049,Call Centre Export


,resident_id,resident_name,age,province,municipality,monthly_household_income,preferred_language,has_smartphone
0,R0001,Resident 1,53,Western Cape,Cape Metro,18211,English,Yes
1,R0002,Resident 2,21,Eastern Cape,Nelson Mandela Bay,24575,English,Yes
2,R0003,Resident 3,21,Gauteng,Ekurhuleni Metro,1500,Sesotho,No
3,R0004,Resident 4,53,Gauteng,Tshwane Metro,10852,English,Yes
4,R0005,Resident 5,53,Eastern Cape,Buffalo City,34858,isiZulu,Yes


,centre_id,centre_name,province,municipality,staff_count,monthly_capacity,operating_hours,active
0,C001,Cape Metro Service Centre 1,Western Cape,Cape Metro,23,867,08h00 to 16h30,True
1,C002,Cape Metro Service Centre 2,Western Cape,Cape Metro,46,1041,8am-4:30pm,False
2,C003,Winelands District Service Centre 3,Western Cape,Winelands District,27,432,08:00-16:30,True
3,C004,Winelands District Service Centre 4,Western Cape,Winelands District,twenty,1222,8am-4:30pm,False
4,C005,Garden Route District Service Centre 5,Western Cape,Garden Route District,45,408,08:00-16:30,True


# Step 3 — Create a source specification (PA0102)

A **source specification** explains what each dataset is and how it will be used.

Important terms:

- **Source:** where the data came from.
- **Format:** CSV, JSON, Excel or another file type.
- **Grain:** what one row represents.
- **Primary key:** a field that should uniquely identify each row.
- **Foreign key:** a field used to link one dataset to another.
- **GIGO:** “Garbage In, Garbage Out.” Poor input data can lead to poor decisions.

Complete the table below by replacing the placeholder text.


In [8]:
source_specification = pd.DataFrame([
    {
        "dataset": "service_requests_raw.csv",
        "format": "CSV",
        "one_row_represents": "Complete this",
        "primary_key": "Complete this",
        "foreign_key": "Complete this",
        "purpose": "Complete this",
        "possible_limitation": "Complete this"
    },
    {
        "dataset": "resident_profiles_raw.csv",
        "format": "CSV",
        "one_row_represents": "Complete this",
        "primary_key": "Complete this",
        "foreign_key": "None",
        "purpose": "Complete this",
        "possible_limitation": "Complete this"
    },
    {
        "dataset": "service_centres_raw.json",
        "format": "JSON",
        "one_row_represents": "Complete this",
        "primary_key": "Complete this",
        "foreign_key": "Complete this",
        "purpose": "Complete this",
        "possible_limitation": "Complete this"
    }
])

display(source_specification)


,dataset,format,one_row_represents,primary_key,foreign_key,purpose,possible_limitation
0,service_requests_raw.csv,CSV,Complete this,Complete this,Complete this,Complete this,Complete this
1,resident_profiles_raw.csv,CSV,Complete this,Complete this,None,Complete this,Complete this
2,service_centres_raw.json,JSON,Complete this,Complete this,Complete this,Complete this,Complete this


## GIGO activity

Describe three examples of how poor-quality data could cause CityConnect to make a bad decision.

1.  Incorrect customer information , outdated bills or incorrect bills may be sent to the wrong people. this could lead to missed payments or services 
2.  Poor planning based on inaccurate resident information, missing values , negative values, inappropriate number of years. if the city makes decison     based on this information , services may be targetted to wrong areas or funding towards incorrect projects 
3.  Incorrect reporting and analysis, duplicate records and repeated records may lead to the city over counting people within the city or servoices.
    manegemnt may believe that there is higher demand that there is actually


# Step 4 — Inspect the data (PA0103)

Never start cleaning immediately. First, learn what is inside the data.

Useful commands:

- `df.shape` — number of rows and columns
- `df.head()` — first five rows
- `df.info()` — column names and data types
- `df.describe()` — numerical summary
- `df.isna().sum()` — missing values
- `df.duplicated().sum()` — duplicate rows
- `df["column"].value_counts()` — category counts


In [9]:
print("REQUEST DATA SHAPE")
print(requests_raw.shape)

print("\nMISSING VALUES")
display(requests_raw.isna().sum().sort_values(ascending=False))

print("\nDUPLICATE ROWS")
print(requests_raw.duplicated().sum())

print("\nDATA TYPES")
display(requests_raw.dtypes)


REQUEST DATA SHAPE
(800, 14)

MISSING VALUES


resolved_at            215
resolution_hours       214
satisfaction_rating    211
device_id                1
service_type             1
created_at               1
submission_channel       0
resident_id              0
request_id               0
status                   0
priority                 0
latitude                 0
longitude                0
source_system            0
dtype: int64


DUPLICATE ROWS
1

DATA TYPES


request_id                 str
resident_id                str
created_at                 str
resolved_at                str
service_type               str
submission_channel         str
priority                   str
status                     str
resolution_hours       float64
satisfaction_rating        str
latitude               float64
longitude              float64
device_id                  str
source_system              str
dtype: object

### Your activity

Repeat the same checks for:
- `residents_raw`
- `centres_raw`

Then write down at least five problems you notice.


In [10]:
# Complete the checks below.

# residents_raw checks
print("REQUEST DATA SHAPE")
print(residents_raw.shape)

print("\nMISSING VALUES")
display(residents_raw.isna().sum().sort_values(ascending=False))

print("\nDUPLICATE ROWS")
print(residents_raw.duplicated().sum())

print("\nDATA TYPES")
display(residents_raw.dtypes)

# centres_raw checks
print("REQUEST DATA SHAPE")
print(centres_raw.shape)

print("\nMISSING VALUES")
display(centres_raw.isna().sum().sort_values(ascending=False))

print("\nDUPLICATE ROWS")
print(centres_raw.duplicated().sum())

print("\nDATA TYPES")
display(centres_raw.dtypes)


REQUEST DATA SHAPE
(400, 8)

MISSING VALUES


resident_name               1
municipality                1
age                         0
resident_id                 0
province                    0
monthly_household_income    0
preferred_language          0
has_smartphone              0
dtype: int64


DUPLICATE ROWS
0

DATA TYPES


resident_id                 str
resident_name               str
age                         str
province                    str
municipality                str
monthly_household_income    str
preferred_language          str
has_smartphone              str
dtype: object

REQUEST DATA SHAPE
(25, 8)

MISSING VALUES


centre_id           0
centre_name         0
province            0
municipality        0
staff_count         0
monthly_capacity    0
operating_hours     0
active              0
dtype: int64


DUPLICATE ROWS
1

DATA TYPES


centre_id              str
centre_name            str
province               str
municipality           str
staff_count         object
monthly_capacity     int64
operating_hours        str
active                bool
dtype: object

## Five problems I found

1.  Duplicate rows 
2.  Missing values
3.  invalid data types 
4.  impossible values 
5.  inconsistent categories 


# Step 5 — Protect the raw data

We keep the original DataFrames unchanged and create copies for cleaning.

This is important because:
- we can compare before and after;
- we can restart if we make a mistake;
- we preserve the original evidence.


In [11]:
requests_clean = requests_raw.copy()
residents_clean = residents_raw.copy()
centres_clean = centres_raw.copy()

print("Cleaning copies created.")


Cleaning copies created.


# Step 6 — Fix mixed data types

Sometimes a number is stored as text, for example:
- `"five"`
- `"R 25,000"`
- `"unknown"`

`pd.to_numeric(..., errors="coerce")` converts valid numbers and changes invalid values to `NaN`.

`NaN` means a value is missing or could not be converted.


In [12]:
# Example: convert satisfaction_rating to a number
requests_clean["satisfaction_rating"] = pd.to_numeric(
    requests_clean["satisfaction_rating"],
    errors="coerce"
)

display(requests_clean["satisfaction_rating"].value_counts(dropna=False).sort_index())


satisfaction_rating
1.0    129
2.0    122
3.0     98
4.0    124
5.0    114
7.0      1
NaN    212
Name: count, dtype: int64

### Your activity

Convert these fields safely:
- `residents_clean["age"]`
- `residents_clean["monthly_household_income"]`
- `centres_clean["staff_count"]`
- `centres_clean["monthly_capacity"]`

For income, first remove `R`, commas and spaces.


In [14]:
# Hint:
# residents_clean["monthly_household_income"] = (
#     residents_clean["monthly_household_income"]
#     .astype(str)
#     .str.replace("R", "", regex=False)
#     .str.replace(",", "", regex=False)
#     .str.strip()
# )
# residents_clean["monthly_household_income"] = pd.to_numeric(
#     residents_clean["monthly_household_income"], errors="coerce"
# )

# Complete your code here.
# Convert age to numeric
residents_clean["age"] = pd.to_numeric(
    residents_clean["age"],
    errors="coerce"
)

# Clean and convert monthly household income
residents_clean["monthly_household_income"] = (
    residents_clean["monthly_household_income"]
    .astype(str)
    .str.replace("R", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip()
)

residents_clean["monthly_household_income"] = pd.to_numeric(
    residents_clean["monthly_household_income"],
    errors="coerce"
)

# Convert staff_count to numeric
centres_clean["staff_count"] = pd.to_numeric(
    centres_clean["staff_count"],
    errors="coerce"
)

# Convert monthly_capacity to numeric
centres_clean["monthly_capacity"] = pd.to_numeric(
    centres_clean["monthly_capacity"],
    errors="coerce"
)


# Step 7 — Convert dates

Dates may be stored in different formats.

`pd.to_datetime()` converts text into a real date-and-time data type.

Using `errors="coerce"` changes unrecognised dates into missing values instead of crashing the notebook.


In [15]:
requests_clean["created_at"] = pd.to_datetime(
    requests_clean["created_at"],
    errors="coerce"
)

requests_clean["resolved_at"] = pd.to_datetime(
    requests_clean["resolved_at"],
    errors="coerce"
)

display(requests_clean[["created_at", "resolved_at"]].head())


,created_at,resolved_at
0,2026-05-26 15:17:00,2026-05-27 11:00:00
1,2026-02-06 20:52:00,NaT
2,2026-02-05 20:44:00,2026-02-08 18:51:00
3,2026-06-07 17:25:00,NaT
4,NaT,NaT


### Check for impossible dates

A request cannot be resolved before it was created.


In [16]:
resolved_before_created = (
    requests_clean["resolved_at"].notna()
    & requests_clean["created_at"].notna()
    & (requests_clean["resolved_at"] < requests_clean["created_at"])
)

print("Rows resolved before creation:", resolved_before_created.sum())
display(requests_clean.loc[resolved_before_created])


Rows resolved before creation: 1


,request_id,resident_id,created_at,resolved_at,service_type,submission_channel,priority,status,resolution_hours,satisfaction_rating,latitude,longitude,device_id,source_system
11,SR00012,R0167,2026-04-07 12:52:00,2025-12-31 10:00:00,Roads,Mobile App,Medium,Closed,69.78,2.0,-29.047625,25.775118,Device-060,Call Centre Export


Explain how you would treat these rows. Do not silently delete them.

_Write your answer here._
 The row should be retained, clearly flagged, investigated and corrected only when reliable evidence is available. This avoids losing potentially useful information while ensuring that incorrect dates do not distort the analysis


# Step 8 — Standardise categories

Categories should use consistent spelling.

For example:
- `electricity`
- `Electricity`
- `electrcity`

These may all mean the same thing.

First inspect the categories.


In [17]:
for column in ["service_type", "submission_channel", "priority", "status"]:
    print("\nCOLUMN:", column)
    display(requests_clean[column].value_counts(dropna=False))



COLUMN: service_type


service_type
Electricity    168
Water          165
Sanitation     163
Waste          154
Roads          148
electrcity       1
NaN              1
Name: count, dtype: int64


COLUMN: submission_channel


submission_channel
Mobile App     226
Web Portal     197
Walk-in        190
Call Centre    186
Whatsapp         1
Name: count, dtype: int64


COLUMN: priority


priority
Medium      331
Low         272
High        151
Critical     45
URGENT        1
Name: count, dtype: int64


COLUMN: status


status
Resolved       367
Closed         218
In Progress    136
Open            78
Done             1
Name: count, dtype: int64

A mapping dictionary tells pandas how to replace incorrect values.


In [18]:
service_type_map = {
    "electrcity": "Electricity",
    "electricity": "Electricity"
}

requests_clean["service_type"] = requests_clean["service_type"].replace(service_type_map)

display(requests_clean["service_type"].value_counts(dropna=False))


service_type
Electricity    169
Water          165
Sanitation     163
Waste          154
Roads          148
NaN              1
Name: count, dtype: int64

### Your activity

Create mapping dictionaries for any other inconsistent categories you discover.

Do not guess blindly. Explain why each replacement is reasonable.



In [19]:
# Complete additional category mappings here.
#Province names, data set contains both Western Cape and WC, WC is known as Western Cape , keeping both makes it seem like its two different places 
province_mapping = {
    "WC": "Western Cape"
}
#Language names, Records use different naming styles, standardisation is important for accuracy
language_mapping = {
    "Zulu": "isiZulu"
}
#Smartphone indicator, Standardising this makes it into one category rather than two seperate categories
smartphone_mapping = {
    "Y": "Yes"
}
#Municipality names, Ekurhuleni metro service has been recorded as Johannesburg , even though categorised in one area while it belongs to another metro
municipality_mapping = {
    "Johannesburg": "Ekurhuleni Metro"
}



# Step 9 — Find missing values

Missing data does not always mean the row must be deleted.

Possible treatments:
- leave it missing;
- fill it using a justified value;
- add a flag;
- remove the row only when essential fields are missing;
- send it back to the source owner for correction.

Count the missing values again after type conversion.


In [20]:
missing_summary = (
    requests_clean.isna()
    .sum()
    .sort_values(ascending=False)
    .to_frame("missing_count")
)

missing_summary["missing_percentage"] = (
    missing_summary["missing_count"] / len(requests_clean) * 100
).round(2)

display(missing_summary)


,missing_count,missing_percentage
resolved_at,215,26.88
resolution_hours,214,26.75
satisfaction_rating,212,26.50
created_at,2,0.25
service_type,1,0.12
device_id,1,0.12
submission_channel,0,0.00
resident_id,0,0.00
request_id,0,0.00
status,0,0.00


### Your activity

Choose treatments for at least three missing-value problems.

Explain:
1. Which field is affected?
   Municipal field, Resident field name and Age Field 
3. How many rows are affected?
   three records in total were affected 
5. What treatment did you choose?
   Municipal field and Resident field :Flag and keep the record, replace missing value with unknown
   Age Field :Change unknown to Nan and fill missing value with median age of dataset
7. Why is that treatment appropriate?
   The municipal cannot be determined with certainty from data
   The resident ID is still available and can uniquely identify the record
   Age field is a numeric value used for analysis.replacing the missing value witht the median age reduces the impact of extreme values and allows the
   record  to remain in the dataset for further analysis 


# Step 10 — Find duplicates

There are two different duplicate checks.

### Complete duplicate row

Every value in the row is repeated.


In [21]:
complete_duplicates = requests_clean.duplicated(keep=False)

print("Rows involved in complete duplicates:", complete_duplicates.sum())
display(requests_clean.loc[complete_duplicates].sort_values("request_id"))


Rows involved in complete duplicates: 2


,request_id,resident_id,created_at,resolved_at,service_type,submission_channel,priority,status,resolution_hours,satisfaction_rating,latitude,longitude,device_id,source_system
249,SR00250,R0129,2026-03-24 19:41:00,NaT,Electricity,Mobile App,Medium,In Progress,NaN,1.0,-28.055828,26.986792,Device-068,Call Centre Export
250,SR00250,R0129,2026-03-24 19:41:00,NaT,Electricity,Mobile App,Medium,In Progress,NaN,1.0,-28.055828,26.986792,Device-068,Call Centre Export


### Duplicate primary key

The request ID should be unique, even if the rest of the row is different.


In [22]:
duplicate_request_ids = requests_clean.duplicated(
    subset=["request_id"],
    keep=False
)

print("Rows with duplicate request IDs:", duplicate_request_ids.sum())
display(requests_clean.loc[duplicate_request_ids].sort_values("request_id"))


Rows with duplicate request IDs: 4


,request_id,resident_id,created_at,resolved_at,service_type,submission_channel,priority,status,resolution_hours,satisfaction_rating,latitude,longitude,device_id,source_system
249,SR00250,R0129,2026-03-24 19:41:00,NaT,Electricity,Mobile App,Medium,In Progress,NaN,1.0,-28.055828,26.986792,Device-068,Call Centre Export
250,SR00250,R0129,2026-03-24 19:41:00,NaT,Electricity,Mobile App,Medium,In Progress,NaN,1.0,-28.055828,26.986792,Device-068,Call Centre Export
300,SR00301,R0151,2026-04-30 18:59:00,2026-05-02 15:50:00,Water,Call Centre,Medium,Closed,44.87,NaN,-27.502040,22.730637,Device-010,Mobile API
301,SR00301,R0398,2026-01-16 14:24:00,NaT,Waste,Walk-in,Critical,In Progress,44.00,NaN,-31.897416,24.335875,Device-026,Call Centre Export


### Your activity

Decide how to treat:
- a completely repeated record;
- two different records with the same request ID.

Explain why these situations should not automatically receive the same treatment.


# Step 11 — Apply business validation rules

A **validation rule** describes what values are acceptable.

Examples:
- age must be between 18 and 100;
- satisfaction rating must be between 1 and 5;
- resolution hours cannot be negative;
- monthly capacity must be positive.

Create Boolean flags. `True` means the row breaks the rule.


In [23]:
requests_clean["invalid_rating"] = (
    requests_clean["satisfaction_rating"].notna()
    & ~requests_clean["satisfaction_rating"].between(1, 5)
)

requests_clean["negative_resolution_hours"] = (
    pd.to_numeric(requests_clean["resolution_hours"], errors="coerce") < 0
)

print("Invalid ratings:", requests_clean["invalid_rating"].sum())
print("Negative resolution hours:", requests_clean["negative_resolution_hours"].sum())


Invalid ratings: 1
Negative resolution hours: 1


### Your activity

Create flags for:
- invalid ages;
- negative household income;
- invalid latitude or longitude;
- non-positive centre capacity;
- resolution dates before creation dates;
- unexpected priority and status categories.


In [24]:
# Complete validation flags here.
# 1. Invalid ages
residents_clean["invalid_age_flag"] = (
    (residents_clean["age_num"] < 0) |
    (residents_clean["age_num"] > 120)
)

# 2. Negative household income
residents_clean["negative_income_flag"] = (
    residents_clean["income_num"] < 0
)

# 3. Invalid latitude or longitude
requests_clean["invalid_coordinates_flag"] = (
    (requests_clean["latitude"] < -90) |
    (requests_clean["latitude"] > 90) |
    (requests_clean["longitude"] < -180) |
    (requests_clean["longitude"] > 180)
)

# 5. Resolution date before creation date
requests_clean["invalid_resolution_date_flag"] = (
    requests_clean["resolved_at"] <
    requests_clean["created_at"]
)

# 6. Unexpected priority categories
valid_priorities = ["Low", "Medium", "High", "Critical"]

requests_clean["invalid_priority_flag"] = (
    ~requests_clean["priority"].isin(valid_priorities)
)

# 6. Unexpected status categories
valid_statuses = ["Open", "In Progress", "Resolved", "Closed"]

requests_clean["invalid_status_flag"] = (
    ~requests_clean["status"].isin(valid_statuses)
)

KeyError: 'age_num'

# Step 12 — Find outliers

An **outlier** is a value far away from most other values.

An outlier may be:
- a data-entry error;
- a system error;
- a real but unusual event.

Therefore, do not delete it automatically.

## IQR method

The IQR method uses the middle 50% of values.


In [25]:
resolution_numeric = pd.to_numeric(
    requests_clean["resolution_hours"],
    errors="coerce"
)

q1 = resolution_numeric.quantile(0.25)
q3 = resolution_numeric.quantile(0.75)
iqr = q3 - q1

lower_limit = q1 - 1.5 * iqr
upper_limit = q3 + 1.5 * iqr

requests_clean["resolution_outlier_iqr"] = (
    resolution_numeric.notna()
    & ((resolution_numeric < lower_limit) | (resolution_numeric > upper_limit))
)

print("IQR lower limit:", lower_limit)
print("IQR upper limit:", upper_limit)
print("Outliers found:", requests_clean["resolution_outlier_iqr"].sum())


IQR lower limit: -20.91875000000001
IQR upper limit: 120.75125
Outliers found: 1


### Second method: percentile rule

Flag values above the 99th percentile.


In [26]:
p99 = resolution_numeric.quantile(0.99)

requests_clean["resolution_outlier_p99"] = (
    resolution_numeric.notna()
    & (resolution_numeric > p99)
)

print("99th percentile:", p99)
display(
    requests_clean.loc[
        requests_clean["resolution_outlier_iqr"]
        | requests_clean["resolution_outlier_p99"],
        ["request_id", "resolution_hours", "priority", "status"]
    ].sort_values("resolution_hours", ascending=False)
)


99th percentile: 112.643


,request_id,resolution_hours,priority,status
38,SR00039,9999.00,Low,Resolved
656,SR00657,120.01,Low,Resolved
178,SR00179,117.72,Low,Closed
19,SR00020,116.42,Low,Resolved
72,SR00073,113.62,Low,Resolved
214,SR00215,112.83,Low,Resolved


### Your judgement

Choose at least three flagged values and explain whether each one is:
- definitely incorrect; SR00039 resolution hours 9999.00. The value is extremely higher than all other records and far above the 99th percentile
- probably incorrect;SR00657 Resolutions hours 120.01. The vlaue is above 99th percintile while possible, it is unusual to warrant an invetsigation
- possibly valid but unusual;SR00179 117.72  This is also higher than the 99th percintile but it is the least compared to the other two, a complex
  service request which could take time to resolve 

Do not remove an outlier without recording the reason.


# Step 13 — Create a data-quality log

A data-quality log is an audit trail of:
- what was wrong;
- how many rows were affected;
- what you did;
- why you did it;
- what risk remains.

Add a row to this table for every major issue.


In [29]:
quality_log_columns = [
    "dataset",
    "field",
    "issue",
    "affected_rows",
    "treatment",
    "justification",
    "remaining_risk"
]

data_quality_log = pd.DataFrame(columns=quality_log_columns)

# Example entry
example_entry = {
    "dataset": "service_requests_raw.csv",
    "field": "service_type",
    "issue": "Spelling variation: electrcity",
    "affected_rows": int((requests_raw["service_type"] == "electrcity").sum()),
    "treatment": "Mapped to Electricity",
    "justification": "The value is an obvious spelling error and Electricity is an approved category.",
    "remaining_risk": "Other unknown spelling errors may still exist."
}

data_quality_log = pd.concat(
    [data_quality_log, pd.DataFrame([example_entry])],
    ignore_index=True
)

display(data_quality_log)


,dataset,field,issue,affected_rows,treatment,justification,remaining_risk
0,service_requests_raw.csv,service_type,Spelling variation: electrcity,1,Mapped to Electricity,The value is an obvious spelling error and Ele...,Other unknown spelling errors may still exist.


### Your activity

Add entries for all the important problems you treated.


In [30]:
# Add your quality-log entries here.
data_quality_log = pd.DataFrame({
    "issue": [
        "Invalid age values",
        "Income stored as text/currency format",
        "Negative household income",
        "Invalid staff_count value",
        "Negative monthly_capacity",
        "Duplicate resident_id",
        "Missing resident name",
        "Missing municipality",
        "Inconsistent province names",
        "Inconsistent language labels"
    ],
    "rows_affected": [
        2,   # unknown, age 150
        1,   # 'R 25,000'
        1,   # -5000
        1,   # 'twenty'
        1,   # -100
        1,   # duplicate R0033
        1,   # missing resident_name
        1,   # blank municipality
        1,   # WC vs Western Cape
        1    # Zulu vs isiZulu
    ],
    "action_taken": [
        "Converted to numeric and set invalid values to NaN",
        "Removed currency symbols and converted to numeric",
        "Flagged/replaced invalid value with NaN",
        "Converted to numeric; invalid value became NaN",
        "Flagged/replaced invalid value with NaN",
        "Investigated and removed/flagged duplicate record",
        "Retained record and flagged missing value",
        "Retained record and flagged missing value",
        "Standardized to 'Western Cape'",
        "Standardized to 'isiZulu'"
    ],
    "reason": [
        "Ages must be numeric and within a valid range",
        "Income analysis requires numeric values",
        "Income cannot be negative",
        "Staff counts must be numeric",
        "Centre capacity cannot be negative",
        "Primary keys should be unique",
        "Avoid unnecessary data loss",
        "Avoid unnecessary data loss",
        "Ensure category consistency",
        "Ensure category consistency"
    ],
    "remaining_risk": [
        "Missing ages may affect age-based analysis",
        "Some income values may still be inaccurate",
        "True income value is unknown",
        "True staff count is unknown",
        "True capacity is unknown",
        "Potential uncertainty about correct duplicate record",
        "Missing name cannot be recovered",
        "Municipality remains unknown",
        "Other undiscovered spelling variations may exist",
        "Other undiscovered language variations may exist"
    ]
})

data_quality_log


,issue,rows_affected,action_taken,reason,remaining_risk
0,Invalid age values,2,Converted to numeric and set invalid values to...,Ages must be numeric and within a valid range,Missing ages may affect age-based analysis
1,Income stored as text/currency format,1,Removed currency symbols and converted to numeric,Income analysis requires numeric values,Some income values may still be inaccurate
2,Negative household income,1,Flagged/replaced invalid value with NaN,Income cannot be negative,True income value is unknown
3,Invalid staff_count value,1,Converted to numeric; invalid value became NaN,Staff counts must be numeric,True staff count is unknown
4,Negative monthly_capacity,1,Flagged/replaced invalid value with NaN,Centre capacity cannot be negative,True capacity is unknown
5,Duplicate resident_id,1,Investigated and removed/flagged duplicate record,Primary keys should be unique,Potential uncertainty about correct duplicate ...
6,Missing resident name,1,Retained record and flagged missing value,Avoid unnecessary data loss,Missing name cannot be recovered
7,Missing municipality,1,Retained record and flagged missing value,Avoid unnecessary data loss,Municipality remains unknown
8,Inconsistent province names,1,Standardized to 'Western Cape',Ensure category consistency,Other undiscovered spelling variations may exist
9,Inconsistent language labels,1,Standardized to 'isiZulu',Ensure category consistency,Other undiscovered language variations may exist


# Step 14 — Check referential integrity and merge data

`resident_id` links service requests to resident profiles.

A **foreign-key problem** occurs when a request refers to a resident who does not exist in the resident table.

Use `merge(..., indicator=True)` to see whether rows matched.


In [31]:
merge_check = requests_clean.merge(
    residents_clean,
    on="resident_id",
    how="left",
    indicator=True,
    suffixes=("_request", "_resident")
)

display(merge_check["_merge"].value_counts())
display(merge_check.loc[merge_check["_merge"] != "both"].head())


_merge
both          796
left_only       6
right_only      0
Name: count, dtype: int64

,request_id,resident_id,created_at,resolved_at,service_type,submission_channel,priority,status,resolution_hours,satisfaction_rating,latitude,longitude,device_id,source_system,invalid_rating,negative_resolution_hours,resolution_outlier_iqr,resolution_outlier_p99,resident_name,age,province,municipality,monthly_household_income,preferred_language,has_smartphone,_merge
77,SR00078,R9999,2026-06-02 17:57:00,2026-06-05 09:34:00,Roads,Call Centre,Medium,Resolved,63.62,NaN,-32.764524,30.489620,Device-007,CRM,False,False,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
244,SR00245,R0034,2026-02-15 22:28:00,2026-02-17 09:23:00,Sanitation,Walk-in,Medium,Resolved,34.93,4.0,-27.446932,26.628640,Device-041,Mobile API,False,False,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
359,SR00359,R0034,2026-03-19 18:00:00,2026-03-21 04:03:00,Water,Web Portal,High,Closed,34.06,2.0,-29.087853,21.040335,Device-004,CRM,False,False,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
393,SR00393,R0034,2026-04-01 14:38:00,2026-04-03 20:26:00,Sanitation,Call Centre,Medium,Resolved,53.80,NaN,-32.601465,24.374648,Device-001,Mobile API,False,False,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
664,SR00663,R0034,2026-04-09 12:38:00,NaT,Electricity,Mobile App,Medium,Open,NaN,3.0,-27.968489,31.263914,Device-074,Call Centre Export,False,False,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only


### Your activity

Explain:
- how many requests did not match a resident; 6 did not match which were left
- why this is a problem; Data qaulity and referential integrity 
- whether you will remove, retain or flag those requests.Retain and flag unmatched rather than to remove because it may contain useful information 

Then create the final merged DataFrame.


In [32]:
final_dataset = merge_check.copy()

# Add an easy-to-understand flag.
final_dataset["resident_match_found"] = final_dataset["_merge"].eq("both")

# Remove the technical merge-indicator column only after creating the flag.
final_dataset = final_dataset.drop(columns=["_merge"])

display(final_dataset.head())


,request_id,resident_id,created_at,resolved_at,service_type,submission_channel,priority,status,resolution_hours,satisfaction_rating,latitude,longitude,device_id,source_system,invalid_rating,negative_resolution_hours,resolution_outlier_iqr,resolution_outlier_p99,resident_name,age,province,municipality,monthly_household_income,preferred_language,has_smartphone,resident_match_found
0,SR00001,R0355,2026-05-26 15:17:00,2026-05-27 11:00:00,Water,Web Portal,High,Resolved,19.73,NaN,-33.913755,31.024099,Device-003,CRM,False,False,False,False,Resident 355,44.0,Western Cape,Garden Route District,15131.0,Sesotho,No,True
1,SR00002,R0301,2026-02-06 20:52:00,NaT,Roads,Walk-in,High,In Progress,NaN,5.0,-32.107892,29.734713,Device-070,Mobile API,False,False,False,False,Resident 301,25.0,Eastern Cape,Nelson Mandela Bay,24791.0,Afrikaans,Yes,True
2,SR00003,R0197,2026-02-05 20:44:00,2026-02-08 18:51:00,Water,Mobile App,Medium,Closed,70.12,1.0,-33.350134,21.480373,Device-044,Call Centre Export,False,False,False,False,Resident 197,59.0,KwaZulu-Natal,King Cetshwayo District,6137.0,isiZulu,No,True
3,SR00004,R0394,2026-06-07 17:25:00,NaT,Waste,Walk-in,Low,In Progress,NaN,4.0,-29.931750,23.174613,Device-056,CRM,False,False,False,False,Resident 394,73.0,Western Cape,Garden Route District,19096.0,isiXhosa,Yes,True
4,SR00005,R0349,NaT,NaT,Water,Call Centre,High,In Progress,NaN,2.0,-27.995956,19.423921,Device-049,Call Centre Export,False,False,False,False,Resident 349,82.0,Gauteng,Tshwane Metro,40226.0,isiZulu,Yes,True


# Step 15 — Write reusable functions (PA0104)

Reproducible code means:
- another person can run it again;
- the same steps are followed every time;
- there are no hidden manual changes.

A function stores a reusable set of instructions.


In [33]:
def missing_value_summary(df):
    summary = df.isna().sum().to_frame("missing_count")
    summary["missing_percentage"] = (
        summary["missing_count"] / len(df) * 100
    ).round(2)
    return summary.sort_values("missing_count", ascending=False)

display(missing_value_summary(final_dataset).head(10))


,missing_count,missing_percentage
resolved_at,215,26.81
resolution_hours,214,26.68
satisfaction_rating,212,26.43
municipality,9,1.12
resident_name,7,0.87
age,7,0.87
province,6,0.75
monthly_household_income,6,0.75
has_smartphone,6,0.75
preferred_language,6,0.75


### Your activity

Create at least two more functions, for example:
- a function to identify duplicate keys;
- a function to validate numeric ranges;
- a function to standardise category text;
- a function to create an IQR outlier flag.


In [34]:
# Write your functions here.
# Function 1: Identify duplicate keys
def find_duplicate_keys(df, key_column):
    return df[df.duplicated(subset=key_column, keep=False)]


# Function 2: Validate numeric ranges
def validate_numeric_range(series, min_value, max_value):
    return ~series.between(min_value, max_value, inclusive="both")


# Function 3: Standardise category text
def standardise_category(series):
    return (
        series.astype(str)
        .str.strip()
        .str.title()
    )


# Function 4: Create an IQR outlier flag
def iqr_outlier_flag(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1

    lower_bound = q1 - (1.5 * iqr)
    upper_bound = q3 + (1.5 * iqr)

    return (series < lower_bound) | (series > upper_bound)


In [35]:
# Find duplicate resident IDs
duplicate_residents = find_duplicate_keys(
    residents_clean,
    "resident_id"
)

display(duplicate_residents)


# Check for invalid ages
invalid_ages = validate_numeric_range(
    residents_clean["age"],
    18,
    100
)

display(residents_clean[invalid_ages])


# Standardise province names
residents_clean["province"] = standardise_category(
    residents_clean["province"]
)


# Flag income outliers
income_outliers = iqr_outlier_flag(
    residents_clean["monthly_household_income"]
)

display(
    residents_clean.loc[
        income_outliers,
        ["resident_id", "monthly_household_income"]
    ]
)

,resident_id,resident_name,age,province,municipality,monthly_household_income,preferred_language,has_smartphone
32,R0033,Resident 33,30.0,Eastern Cape,Nelson Mandela Bay,28453,Afrikaans,No
33,R0033,Resident 34,24.0,Eastern Cape,OR Tambo District,25290,English,Yes


,resident_id,resident_name,age,province,municipality,monthly_household_income,preferred_language,has_smartphone
7,R0008,Resident 8,NaN,KwaZulu-Natal,King Cetshwayo District,10717,Sesotho,Yes
88,R0089,Resident 89,150.0,Eastern Cape,Nelson Mandela Bay,14412,Afrikaans,Yes


,resident_id,monthly_household_income
17,R0018,49493


# Step 16 — Final validation

Assertions are automatic checks. The notebook stops if an important rule fails.

Only use an assertion after you have decided what the final rule should be.


In [38]:
# Example checks. Adapt them to your final cleaning decisions.

assert len(final_dataset) > 0, "The final dataset is empty."
assert "request_id" in final_dataset.columns, "request_id is missing."
assert "resident_match_found" in final_dataset.columns, "Merge flag is missing."

print("Basic validation checks passed.")


Basic validation checks passed.


### Your activity

Add assertions for:
- unique request IDs, if that is your final rule;
- no negative resolution hours;
- ratings only between 1 and 5 when present;
- required fields being present.


In [61]:
# Add final assertions here.
# 1. Dataset must not be empty
assert len(final_dataset) > 0, \
    "The final dataset is empty."

# 2. Required columns must be present
required_columns = [
    "request_id",
    "resident_id",
    "resident_match_found",
    "resolution_hours",
    "satisfaction_rating"
]

for column in required_columns:
    assert column in final_dataset.columns, \
        f"Required column {column} is missing."

# 3. Request IDs must be unique
duplicates = final_dataset[
    final_dataset["request_id"].duplicated(keep=False)
]
display(duplicates.sort_values("request_id"))

# 4. Resolution hours must not be negative
negative_hours = final_dataset[
    final_dataset["resolution_hours"] < 0
]

display(negative_hours)

# 5. Ratings must be between 1 and 5 when present
# Ratings must be between 1 and 5 when present
invalid_ratings = final_dataset[
    ~final_dataset["satisfaction_rating"].between(1, 5)
    & final_dataset["satisfaction_rating"].notna()
]

display(invalid_ratings)

# 6. Required fields must not contain missing values
required_fields = [
    "request_id",
    "resident_id",
    "resident_match_found"
]

for column in required_fields:
    assert final_dataset[column].notna().all(), \
        f"Missing values found in {column}."

print("All final validation checks passed!")

,request_id,resident_id,created_at,resolved_at,service_type,submission_channel,priority,status,resolution_hours,satisfaction_rating,latitude,longitude,device_id,source_system,invalid_rating,negative_resolution_hours,resolution_outlier_iqr,resolution_outlier_p99,resident_name,age,province,municipality,monthly_household_income,preferred_language,has_smartphone,resident_match_found
249,SR00250,R0129,2026-03-24 19:41:00,NaT,Electricity,Mobile App,Medium,In Progress,NaN,1.0,-28.055828,26.986792,Device-068,Call Centre Export,False,False,False,False,Resident 129,47.0,Eastern Cape,Nelson Mandela Bay,36761.0,Sesotho,Yes,True
250,SR00250,R0129,2026-03-24 19:41:00,NaT,Electricity,Mobile App,Medium,In Progress,NaN,1.0,-28.055828,26.986792,Device-068,Call Centre Export,False,False,False,False,Resident 129,47.0,Eastern Cape,Nelson Mandela Bay,36761.0,Sesotho,Yes,True
300,SR00301,R0151,2026-04-30 18:59:00,2026-05-02 15:50:00,Water,Call Centre,Medium,Closed,44.87,NaN,-27.502040,22.730637,Device-010,Mobile API,False,False,False,False,Resident 151,23.0,Gauteng,Tshwane Metro,12358.0,Afrikaans,No,True
301,SR00301,R0398,2026-01-16 14:24:00,NaT,Waste,Walk-in,Critical,In Progress,44.00,NaN,-31.897416,24.335875,Device-026,Call Centre Export,False,False,False,False,Resident 398,66.0,Western Cape,Winelands District,17473.0,Afrikaans,No,True
308,SR00309,R0033,2026-05-19 18:19:00,2026-05-21 22:44:00,Roads,Web Portal,Medium,Closed,52.43,4.0,-25.790805,18.524931,Device-008,Mobile API,False,False,False,False,Resident 33,30.0,Eastern Cape,Nelson Mandela Bay,28453.0,Afrikaans,No,True
309,SR00309,R0033,2026-05-19 18:19:00,2026-05-21 22:44:00,Roads,Web Portal,Medium,Closed,52.43,4.0,-25.790805,18.524931,Device-008,Mobile API,False,False,False,False,Resident 34,24.0,Eastern Cape,OR Tambo District,25290.0,English,Yes,True
503,SR00503,R0033,2026-04-13 17:51:00,2026-04-16 05:06:00,Electricity,Mobile App,Medium,Closed,59.26,4.0,-30.196220,26.193019,Device-037,Call Centre Export,False,False,False,False,Resident 33,30.0,Eastern Cape,Nelson Mandela Bay,28453.0,Afrikaans,No,True
504,SR00503,R0033,2026-04-13 17:51:00,2026-04-16 05:06:00,Electricity,Mobile App,Medium,Closed,59.26,4.0,-30.196220,26.193019,Device-037,Call Centre Export,False,False,False,False,Resident 34,24.0,Eastern Cape,OR Tambo District,25290.0,English,Yes,True


,request_id,resident_id,created_at,resolved_at,service_type,submission_channel,priority,status,resolution_hours,satisfaction_rating,latitude,longitude,device_id,source_system,invalid_rating,negative_resolution_hours,resolution_outlier_iqr,resolution_outlier_p99,resident_name,age,province,municipality,monthly_household_income,preferred_language,has_smartphone,resident_match_found
22,SR00023,R0235,2026-06-08 10:52:00,2026-06-09 03:31:00,Waste,Call Centre,High,Resolved,-4.5,2.0,-31.543194,20.80771,Device-066,Mobile API,False,True,False,False,Resident 235,74.0,Western Cape,Cape Metro,32599.0,Sesotho,Yes,True


,request_id,resident_id,created_at,resolved_at,service_type,submission_channel,priority,status,resolution_hours,satisfaction_rating,latitude,longitude,device_id,source_system,invalid_rating,negative_resolution_hours,resolution_outlier_iqr,resolution_outlier_p99,resident_name,age,province,municipality,monthly_household_income,preferred_language,has_smartphone,resident_match_found
44,SR00045,R0035,2026-01-04 12:13:00,2026-01-05 00:21:00,Water,Walk-in,High,Closed,12.15,7.0,-29.840874,23.991418,Device-011,Call Centre Export,True,False,False,False,Resident 35,61.0,Eastern Cape,OR Tambo District,19804.0,isiXhosa,Yes,True


All final validation checks passed!


# Step 17 — Save the outputs

Use `to_csv()` to save the final tables.

`index=False` prevents pandas from adding an unnecessary row-number column.


In [63]:
final_dataset.to_csv(
    OUTPUT_FOLDER / "clean_service_requests.csv",
    index=False
)

data_quality_log.to_csv(
    OUTPUT_FOLDER / "data_quality_log.csv",
    index=False
)

source_specification.to_csv(
    OUTPUT_FOLDER / "source_specification.csv",
    index=False
)

print("Files saved inside:", OUTPUT_FOLDER.resolve())


Files saved inside: C:\Users\23nto\Downloads\PM6_Beginner_Guided_Tutorial_Assignment_Pack (1)\outputs


# Step 18 — Present preliminary findings

Now that the data has been prepared, calculate simple summaries to motivate the business problem.

Examples:
- request count by service type;
- median resolution hours by service type;
- median resolution hours by submission channel;
- percentage of unmatched residents;
- percentage of requests with missing satisfaction ratings.

Do not perform complex modelling. The focus of this assignment is data preparation.


In [64]:
# Example
service_summary = (
    final_dataset.groupby("service_type", dropna=False)
    .agg(
        request_count=("request_id", "count"),
        median_resolution_hours=("resolution_hours", "median")
    )
    .sort_values("median_resolution_hours", ascending=False)
)

display(service_summary)


,request_count,median_resolution_hours
service_type,,
Electricity,170,54.850
Water,165,53.685
Sanitation,163,51.600
Roads,149,46.390
Waste,154,44.700
NaN,1,NaN


## My three preliminary observations

1. Electricity service requests have the longest median resolution time, this suggests that service category has delays compared to others with a
   median hours of 54.85
2. Roads and watse requests are solved quicker compared to other requests , the median resolution time is 44.70 while roads is 46.39 showing faster
   turn around times 
3. Data quality issue in service request field, one record has NaN service typeand no median resolution time which may hint incomplete data


# Final readiness conclusion

Write 250–400 words in plain language.

Answer:
1. Is the data now ready for analysis?
2. What were the biggest quality problems?
3. Which cleaning choices may affect the results?
4. What problems remain unresolved?
5. What should CityConnect improve in its data-capture process?

_Write your conclusion here._
The data is now ready for use as most of the data qaulity issues have been solved, documented and cleaned using repeatable code. The data was inspected , data stypes were corrected , category values was standardised not limited to this but duplicated records were looked into. Service requests were successfully merged witht the resident data. Data cleaning was documented in a data qaulity log making the work reproducible and easy to audit. This data is suitable for the following service request delays, service categories , municipal and reporting channels 

Qaulity probelsm included missing values , duplicatesm identifiers and inconsistent category labels, mixed data dypes , invalid numeric values, referential integrity issues. E.G some categories were stored as text such as age,staff count , household income , negative income value which does not exist as someone cannot earn negative. Some instance had duplicated Resident ID and requests which did not match a profile. These may lead to inaccurate results if they had not beeen corrected 

Cleaning decisions may affect the final results. Invalid values were converted to missing values, duplicate records were removed and flagged. Inconsistent categories were standardised. These decisions made was to improve data quality but may affect summary statistics slightly, not summary statistics but other components such as  averages , counts , trend analysis because some information is no longer available or has been grouped into standard categories 

Few issues remain unresolved , some records still have missing values , especially the orginal information. There are also unmacthed records between dataset and some field where the true value is unknown. while these issues do not prevent further analysis  they should be taken into consideration when intepreting the results.

To improve future data qaulity, Cityconnect need to introduce stronger validation rules during data capturing. Mandatory fields need to contain information and not remain blank, numeric fields should only take numbers not text and unique identifiers  should automatically check for duplicates. Some fields need to use drop down options rather than taking texts. Regular data quality monitoring and staff training is essential as this would allow errors to be detected early  before being captured on the system

Overall the data is sufficiently clean and reliable for analysis but the remaining limitations should be acknowledged when reporting findings and making business decisions 


# Final checklist

- [ ] I explained the business problem.
- [ ] I loaded all three files.
- [ ] I created a source specification.
- [ ] I inspected all datasets.
- [ ] I kept the original data unchanged.
- [ ] I corrected mixed data types.
- [ ] I checked dates and categories.
- [ ] I checked missing values and duplicates.
- [ ] I created validation flags.
- [ ] I used two outlier methods.
- [ ] I checked foreign-key matches.
- [ ] I created a data-quality log.
- [ ] I used NumPy and pandas.
- [ ] I created reusable functions.
- [ ] My notebook runs from top to bottom.
- [ ] I exported all three output files.
